# 03 — Evaluation
## Brain Tumour Detection — Final Project
=====================================

Topics covered:
  1.  Opening the test set, once
  2.  Headline accuracy, and why it is not enough
  3.  The confusion matrix — which class is being missed
  4.  Precision, recall and F1 per class
  5.  Confidence intervals — how much of the number is signal
  6.  ROC and AUC — performance without a fixed threshold
  7.  Calibration — is the model unsure when it is wrong
  8.  The mistakes, looked at individually
  9.  Feature maps — what the layers learned
  10. Grad-CAM — is it right for the right reason
  11. The shortcut audit — testing what Grad-CAM suggested
  12. Summary and honest limitations

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from src import config, data, engine, metrics, viz
from src.config import CACHE_DIR, CKPT_PATH, DEVICE, FACE, PALETTE, TRAIN_DIR

test_img = np.load(CACHE_DIR / "test_img.npy")
test_lab = np.load(CACHE_DIR / "test_lab.npy")

model, ckpt = engine.load_checkpoint(CKPT_PATH)
CLASSES   = ckpt["classes"]
MEAN, STD = ckpt["norm"]
print(f"loaded checkpoint from epoch {ckpt['epoch']}  "
      f"(val loss {ckpt['val_loss']:.4f}, val acc {ckpt['val_acc']:.4f})")
print(f"preprocessing travelled with it: img_size {ckpt['img_size']}, "
      f"MEAN {MEAN:.4f}, STD {STD:.4f}")

loaded checkpoint from epoch 56  (val loss 0.1217, val acc 0.9833)
preprocessing travelled with it: img_size 128, MEAN 0.2250, STD 0.1901


In [2]:
# 1. OPENING THE TEST SET, ONCE
"""
This is the first time these 1,497 images are used for anything. They were not
involved in choosing the architecture, the learning rate, the augmentation, the
stopping epoch or the normalisation constants — all of that was decided against
the validation split in notebook 02.

That distinction is the whole value of the number below. A test set that has
been peeked at during development stops being a test set and becomes a second
validation set with an optimistic bias nobody can measure.

The preprocessing is taken from the checkpoint rather than recomputed, so the
model cannot accidentally be scored under a different pipeline than it trained
under.
"""
eval_tf = data.make_transforms(MEAN, STD, augment=False, img_size=ckpt["img_size"])
test_ds = data.CachedDataset(test_img, test_lab, eval_tf)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False)

y_true, y_pred, probs = metrics.predict(model, test_loader)
print(f"test images scored: {len(y_true)}")
assert len(y_true) == len(test_lab), "loader dropped images — check drop_last"
print(f"every test image scored, none dropped  [OK]")

test images scored: 1497
every test image scored, none dropped  [OK]


In [3]:
# 2. HEADLINE ACCURACY, AND WHY IT IS NOT ENOUGH
"""
Accuracy is the number that goes in an abstract, and on its own it is close to
useless for a medical classifier. It averages over classes, so a model that
handles three classes perfectly and fails the fourth can still post a
respectable figure. It also treats every error as equally bad, when calling a
tumour healthy and calling a healthy scan a tumour have completely different
consequences.

It is reported first because it is expected, and then immediately decomposed.
"""
acc = (y_true == y_pred).mean()
print(f"test accuracy: {acc:.4f}  ({(y_true == y_pred).sum()} of {len(y_true)})")
print(f"\nfor reference:")
print(f"  chance                     0.2500")
print(f"  validation (notebook 02)   {ckpt['val_acc']:.4f}   <- selection metric, biased upward")
print(f"  test (this number)         {acc:.4f}")

if acc > 0.995:
    print("\n  WARNING: above 99.5% on this dataset is more likely leakage than skill.")
elif acc < 0.30:
    print("\n  WARNING: near chance — suspect a preprocessing or label bug, not a weak model.")

test accuracy: 0.9439  (1413 of 1497)

for reference:
  chance                     0.2500
  validation (notebook 02)   0.9833   <- selection metric, biased upward
  test (this number)         0.9439


In [4]:
# 3. THE CONFUSION MATRIX
"""
The confusion matrix answers the question accuracy hides: not how many were
wrong, but which class got mistaken for which.

The row-normalised panel is the one to read. Each row sums to one, so the
diagonal is per-class recall and the off-diagonal entries show where that
class's scans actually went.

There is a specific expectation worth stating in advance, because a result that
matches a prediction made beforehand is worth more than one explained
afterwards: glioma and meningioma should be the dominant confusion. Both are
masses that present as bright regions; what separates them is whether the
tumour arises inside brain tissue or from the meninges covering it, which is a
distinction about margin and position rather than intensity. If instead the
model were confusing notumor with anything, that would point at a bug.
"""
cm = metrics.confusion(y_true, y_pred, len(CLASSES))
viz.plot_confusion(cm, CLASSES, name="confusion_matrix.png")

off = [(cm[i, j], CLASSES[i], CLASSES[j])
       for i in range(len(CLASSES)) for j in range(len(CLASSES)) if i != j]
off.sort(reverse=True)
print("\nlargest confusions:")
for n, true_c, pred_c in off[:4]:
    if n:
        print(f"  {n:>4} {true_c} scans predicted as {pred_c}")

  saved -> outputs/confusion_matrix.png

largest confusions:
    33 glioma scans predicted as meningioma
    31 glioma scans predicted as notumor
     5 meningioma scans predicted as pituitary
     4 glioma scans predicted as pituitary


In [5]:
# 4. PRECISION, RECALL AND F1 PER CLASS
"""
Three numbers per class, and they answer different questions.

Precision: of the scans we called glioma, how many were. This is the number
that matters for unnecessary follow-up — every false positive is a patient sent
for further investigation they did not need.

Recall: of the scans that really were glioma, how many we caught. This is the
number that matters clinically, because the complement of recall is the missed
diagnosis rate.

F1 is their harmonic mean, which punishes a model that buys one by sacrificing
the other.

Macro average weights every class equally; weighted average weights by support.
With balanced classes they nearly coincide here, which is itself worth noting —
on an imbalanced dataset the gap between them is where a misleading headline
number hides.
"""
rows = metrics.per_class_report(y_true, y_pred, CLASSES)
metrics.print_report(rows)

worst = min(rows[:len(CLASSES)], key=lambda r: r["recall"])
print(f"\nweakest class by recall: {worst['class']} at {worst['recall']:.4f}")
print(f"-> {(1-worst['recall'])*100:.1f}% of {worst['class']} scans were missed")

class           precision   recall       f1  support
----------------------------------------------------
glioma             0.9822   0.8300   0.8997      400
meningioma         0.8910   0.9630   0.9256      297
notumor            0.9217   1.0000   0.9592      400
pituitary          0.9777   0.9875   0.9826      400
----------------------------------------------------
macro avg          0.9431   0.9451   0.9418     1497
weighted avg       0.9467   0.9439   0.9429     1497

weakest class by recall: glioma at 0.8300
-> 17.0% of glioma scans were missed


In [6]:
# 5. CONFIDENCE INTERVALS
"""
A test accuracy is a point estimate computed on one particular sample of 1,497
images. Quoting it to four decimal places implies a precision the sample size
does not support.

Bootstrapping makes the uncertainty explicit: resample the test set with
replacement a thousand times, score each resample, and read off the 2.5th and
97.5th percentiles. The width of that interval is how much of the reported
number is the model and how much is which images happened to be in the split.

Per-class recall gets the same treatment, and its intervals are wider because
each rests on roughly 300 images rather than 1,497.
"""
point, lo, hi = metrics.bootstrap_ci(y_true, y_pred)
print(f"overall accuracy   {point:.4f}   95% CI [{lo:.4f}, {hi:.4f}]   "
      f"width {hi-lo:.4f}\n")

print(f"{'class':<14}{'recall':>9}{'95% CI':>20}{'support':>9}")
print("-" * 52)
ci_lows = []
for i, name in enumerate(CLASSES):
    p, l, h = metrics.bootstrap_ci(y_true, y_pred, class_idx=i)
    ci_lows.append(l)
    print(f"{name:<14}{p:>9.4f}   [{l:.4f}, {h:.4f}]{(y_true==i).sum():>9}")

print(f"\n-> the defensible claim is that recall exceeds {min(ci_lows):.3f} for every")
print("   class, not that it equals the point estimates above.")

overall accuracy   0.9439   95% CI [0.9325, 0.9539]   width 0.0214

class            recall              95% CI  support
----------------------------------------------------
glioma           0.8300   [0.7924, 0.8650]      400
meningioma       0.9630   [0.9394, 0.9832]      297
notumor          1.0000   [1.0000, 1.0000]      400
pituitary        0.9875   [0.9750, 0.9975]      400

-> the defensible claim is that recall exceeds 0.792 for every
   class, not that it equals the point estimates above.


In [7]:
# 6. ROC AND AUC
"""
Everything so far scored the model at one operating point: whichever class has
the highest softmax probability wins. That is a choice, not a law, and a
clinical deployment might well prefer a different one — flagging anything with
more than a 20% tumour probability for human review, for instance.

An ROC curve sweeps that threshold across its whole range, and the area under
it summarises performance without committing to any single point. Because this
is a four-class problem the curve is computed one-vs-rest: each class against
all others in turn.

AUC has a clean interpretation. It is the probability that a randomly chosen
scan of the target class is scored higher for that class than a randomly chosen
scan that is not.
"""
curves, aucs, macro_auc = metrics.roc_ovr(y_true, probs, len(CLASSES))
viz.plot_roc(curves, aucs, macro_auc, CLASSES, name="roc_curves.png")

print(f"\n{'class':<14}{'AUC':>10}")
print("-" * 24)
for i, name in enumerate(CLASSES):
    print(f"{name:<14}{aucs[i]:>10.4f}")
print("-" * 24)
print(f"{'macro':<14}{macro_auc:>10.4f}")

  saved -> outputs/roc_curves.png

class                AUC
------------------------
glioma            0.9585
meningioma        0.9934
notumor           0.9950
pituitary         0.9996
------------------------
macro             0.9866


In [8]:
# 7. CALIBRATION
"""
A model that is 99% confident on every prediction, including the wrong ones,
carries no usable uncertainty signal. That matters here in a practical way: any
realistic use of a system like this routes low-confidence cases to a human, and
that routing only works if low confidence actually predicts being wrong.

The histogram splits maximum softmax probability by whether the prediction was
correct. What we want is the incorrect distribution sitting to the left of the
correct one. Confident mistakes — errors up at 0.99 — are the dangerous cases,
because nothing in the output would flag them for review.

One thing to keep in mind reading the numbers below: this model trains with
label smoothing at 0.1, which notebook 04 selected. Smoothing deliberately caps
how confident the model is rewarded for being, so the whole confidence
distribution is compressed and absolute cutoffs like 0.99 no longer mean what
they would under a one-hot target. Referral is therefore reported as a
workload fraction — refer the least confident 5%, 10%, 20% — which is both
robust to that compression and closer to how a triage budget is actually set.
"""
viz.plot_confidence(y_true, y_pred, probs, name="confidence.png")

conf = probs.max(axis=1)
right, wrong = conf[y_true == y_pred], conf[y_true != y_pred]
print(f"\nmean confidence when correct   {right.mean():.4f}")
print(f"mean confidence when wrong     {wrong.mean() if len(wrong) else float('nan'):.4f}")
print(f"separation                     {right.mean() - (wrong.mean() if len(wrong) else 0):.4f}")
print(f"max confidence observed        {conf.max():.4f}   "
      f"(label smoothing caps this below 1.0)")
if len(wrong):
    hi = np.quantile(conf, 0.75)
    print(f"\nerrors in the top confidence quartile (>{hi:.3f}): "
          f"{(wrong > hi).sum()} of {len(wrong)}")
    print(f"-> these are the cases a confidence threshold would fail to catch")

    print(f"\n{'refer least confident':<24}{'scans kept':>12}{'acc on kept':>14}"
          f"{'errors caught':>15}")
    print("-" * 65)
    for frac in (0.05, 0.10, 0.20):
        keep = conf > np.quantile(conf, frac)
        caught = (y_true[~keep] != y_pred[~keep]).sum()
        print(f"{frac:>19.0%}{keep.mean():>16.1%}"
              f"{(y_true[keep] == y_pred[keep]).mean():>14.4f}"
              f"{caught / len(wrong):>15.1%}")
    print("\n-> a referral budget is only worth having if the last column beats")
    print("   the first: sending 10% of scans to a human should catch")
    print("   substantially more than 10% of the errors.")

  saved -> outputs/confidence.png

mean confidence when correct   0.9216
mean confidence when wrong     0.7835
separation                     0.1382
max confidence observed        0.9517   (label smoothing caps this below 1.0)

errors in the top confidence quartile (>0.931): 6 of 84
-> these are the cases a confidence threshold would fail to catch

refer least confident     scans kept   acc on kept  errors caught
-----------------------------------------------------------------
                 5%           95.0%        0.9705          50.0%
                10%           90.0%        0.9837          73.8%
                20%           80.0%        0.9866          81.0%

-> a referral budget is only worth having if the last column beats
   the first: sending 10% of scans to a human should catch
   substantially more than 10% of the errors.


In [9]:
# 8. THE MISTAKES, LOOKED AT INDIVIDUALLY
"""
Aggregate metrics describe errors; they do not explain them. Sorting the
mistakes by confidence and looking at the worst ones is how you find out
whether the model is failing on genuinely hard cases or on something
systematic — a scan orientation it never saw, an artefact, a mislabelled image.

This dataset is known to contain some label noise, particularly in the glioma
class, which was reported as mis-categorised in the source data. Some of what
appears below may be the model being right and the label being wrong.
"""
viz.plot_misclassified(test_ds, y_true, y_pred, probs, CLASSES, MEAN, STD,
                       name="misclassified.png", n=12)

n_wrong = int((y_true != y_pred).sum())
print(f"\n{n_wrong} mistakes out of {len(y_true)} test images")
if n_wrong:
    pairs = {}
    for t, p in zip(y_true[y_true != y_pred], y_pred[y_true != y_pred]):
        pairs[(CLASSES[t], CLASSES[p])] = pairs.get((CLASSES[t], CLASSES[p]), 0) + 1
    for (t, p), n in sorted(pairs.items(), key=lambda kv: -kv[1])[:5]:
        print(f"  {n:>4}  {t} -> {p}")

  saved -> outputs/misclassified.png

84 mistakes out of 1497 test images
    33  glioma -> meningioma
    31  glioma -> notumor
     5  meningioma -> pituitary
     4  glioma -> pituitary
     3  meningioma -> notumor


In [10]:
# 9. FEATURE MAPS — WHAT THE LAYERS LEARNED
"""
Day 3 established what a convolution computes. This shows what these particular
convolutions ended up computing after training on real scans.

Block 1 should be recognisable: edges, gradients, texture. Those filters are
generic, which is precisely why transfer learning works at all — early layers
of almost any vision model converge to something similar.

Block 4 should not be recognisable. By that depth the maps are sparse and
spatially coarse, responding to configurations of features rather than to
pixels. That progression from concrete to abstract is the thing a convolutional
stack is for, and seeing it confirms the network learned a hierarchy rather
than memorising intensities.
"""
glioma_i = int(np.where(test_lab == CLASSES.index("glioma"))[0][0])
x, _ = test_ds[glioma_i]
viz.plot_feature_maps(model, x, MEAN, STD, name="feature_maps.png", n=16)
print(f"\nshowing 16 of 32 block-1 channels and 16 of 256 block-4 channels")
print(f"for one {CLASSES[test_lab[glioma_i]]} scan")

  saved -> outputs/feature_maps.png

showing 16 of 32 block-1 channels and 16 of 256 block-4 channels
for one glioma scan


In [11]:
# 10. GRAD-CAM — IS IT RIGHT FOR THE RIGHT REASON
"""
This is the figure that matters most in a medical imaging report, because it
addresses the failure mode that accuracy cannot detect.

A model can reach high accuracy by learning something real about tumours, or by
learning an artefact that happens to correlate with the label — a scanner
signature, a text annotation burned into the corner, a difference in how one
class was cropped. Both look identical in a confusion matrix. They look
completely different here.

Grad-CAM takes the final convolutional feature map, weights each channel by how
strongly the predicted class score responds to it, and sums. The result is a
coarse map of which regions drove the decision. The architecture makes this
almost free: it is already conv -> global average pool -> linear, which is the
structure Grad-CAM was designed around.

What we want to see is heat on the lesion. Heat on the skull margin or in the
background would mean the accuracy above is not measuring what it appears to.
"""
samples = []
for c, name in enumerate(CLASSES):
    correct = np.where((test_lab == c) & (y_true == y_pred))[0]
    wrong   = np.where((test_lab == c) & (y_true != y_pred))[0]
    picks = [(int(i), "") for i in correct[:2]]
    picks.append((int(wrong[0]), "<- error") if len(wrong) else (int(correct[2]), ""))
    samples += [(test_ds[i][0], int(test_lab[i]), tag) for i, tag in picks]

viz.plot_cam_grid(model, samples, CLASSES, MEAN, STD, name="grad_cam.png")
print(f"\none row per class: two correct predictions and one error where available")

  saved -> outputs/grad_cam.png

one row per class: two correct predictions and one error where available


In [12]:
# 11. THE SHORTCUT AUDIT
"""
Grad-CAM raised a suspicion; this section tests it.

The two figures above do not look like a model reading anatomy. The heat sits
on the skull margin rather than on lesions, and the most confident mistakes are
large, unmistakable tumours called notumor at 1.00 confidence — not the subtle
cases a genuinely struggling classifier would fail on.

There is a structural reason to expect this. This Kaggle dataset is a merge of
three separate collections: the three tumour classes come from figshare and
SARTAJ, the healthy scans from Br35H. Those collections were digitised at
different native resolutions, and the split is nearly clean — almost every
tumour scan is 512x512 and almost no healthy scan is.

That makes source a near-perfect predictor of label, and resizing does not
remove it. Every image is resized to 128px, so the literal dimensions are gone,
but a 512 -> 128 downscale and a 225 -> 128 downscale leave different amounts
of blur and different JPEG artefacts. The signature survives in the texture.

If the network learned that signature instead of anatomy, accuracy will
collapse on exactly the images where the signature points the wrong way: tumour
scans that happen not to be 512x512. That is a falsifiable prediction, and the
test set contains enough of those images to check it.

The training split is checked first, because it decides whether the network had
any choice in the matter.
"""
from torchvision.datasets import ImageFolder
from PIL import Image
from src.config import TEST_DIR

_train = ImageFolder(str(TRAIN_DIR)).samples
tr512 = np.array([Image.open(p).size == (512, 512) for p, _ in _train])
tr_lab = np.array([y for _, y in _train])

print("native resolution in the TRAINING split — what the model learned from:")
print(f"  {'class':<12}{'512x512':>9}{'other':>8}")
for c, nm in enumerate(CLASSES):
    m = tr_lab == c
    print(f"  {nm:<12}{(m & tr512).sum():>9}{(m & ~tr512).sum():>8}")

print("""
Read that table before anything else. Every single training glioma is 512x512,
and almost every training notumor is not. So on the data the model was given,
"this file is not 512x512" implies "notumor" with essentially no exceptions.

That reframes the problem. The network did not fail to learn the rule we wanted;
it learned the rule that was actually optimal on the training set. A shortcut is
not a bug in the model, it is a property of the data.

It also explains why validation never caught this. The validation split is
carved out of Training/, so it inherits the same clean correlation and scores
the shortcut as correct every time. Only the test set, assembled differently,
contains tumour scans in the other style.
""")

_samples = data.drop_augmented(ImageFolder(str(TEST_DIR)).samples)
assert len(_samples) == len(y_true), "file order does not match the cache"
is512 = np.array([Image.open(p).size == (512, 512) for p, _ in _samples])
correct = y_true == y_pred

print(f"native resolution in the TEST split:")
print(f"  {'class':<12}{'512x512':>9}{'other':>8}")
for c, nm in enumerate(CLASSES):
    m = y_true == c
    print(f"  {nm:<12}{(m & is512).sum():>9}{(m & ~is512).sum():>8}")

print(f"\naccuracy split by native resolution:")
print(f"  {'class':<12}{'acc|512':>9}{'acc|other':>11}{'gap':>9}{'n_other':>9}")
print("  " + "-" * 50)
gaps = []
for c, nm in enumerate(CLASSES):
    m = y_true == c
    a, b = m & is512, m & ~is512
    av = correct[a].mean() if a.sum() else float('nan')
    bv = correct[b].mean() if b.sum() else float('nan')
    gaps.append(av - bv)
    print(f"  {nm:<12}{av:>9.3f}{bv:>11.3f}{av-bv:>9.3f}{b.sum():>9}")

NOTUMOR = CLASSES.index("notumor")
tum = y_true != NOTUMOR
print(f"\ntumour scans called notumor:")
print(f"  when the file was 512x512     "
      f"{(y_pred[tum & is512] == NOTUMOR).mean():.3f}")
print(f"  when it was any other size    "
      f"{(y_pred[tum & ~is512] == NOTUMOR).mean():.3f}")

fig, ax = viz.styled_fig(figsize=(7.5, 4))
w = 0.38
xs = np.arange(len(CLASSES))
a512 = [correct[(y_true == c) & is512].mean() for c in range(len(CLASSES))]
aoth = [correct[(y_true == c) & ~is512].mean() if ((y_true == c) & ~is512).sum()
        else np.nan for c in range(len(CLASSES))]
ax.bar(xs - w/2, a512, w, color=PALETTE["train"], label="native 512x512")
ax.bar(xs + w/2, aoth, w, color=PALETTE["val"],   label="any other size")
ax.set_xticks(xs); ax.set_xticklabels(CLASSES, fontsize=9)
ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.05)
ax.axhline(0.25, color='k', ls='--', lw=1, alpha=0.6, label="chance")
ax.set_title("Accuracy by native file resolution — the shortcut",
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8); ax.set_facecolor(FACE)
plt.tight_layout(); viz.save(fig, "shortcut_audit.png")

worst_gap = np.nanmax(gaps)
print(f"\nlargest per-class gap: {worst_gap:.3f}")
if worst_gap > 0.20:
    print("""
-> CONFIRMED. The model reads source signature, not anatomy, and the headline
   accuracy overstates what it knows about tumours.

   Two things follow, and the second is the one worth putting in the report.

   First, this is not fixable by augmentation. An augmentation can teach
   invariance to a nuisance variable the model has seen vary; it cannot invent
   training examples of a class in a style that occurs zero times under that
   label. A resample augmentation was tried specifically to destroy the blur
   signature and moved the glioma gap not at all — a negative result, and the
   right one, because blur was never the mechanism. The mechanism is a missing
   region of the training distribution.

   Second, no amount of held-out evaluation on this dataset would have caught
   it. Train, validation and test all inherit the same correlation. It was
   found by looking at the pictures — Grad-CAM pointing at the skull, and an
   error gallery full of obvious tumours labelled healthy with total
   confidence. That is the argument for those figures existing at all.""")
else:
    print("-> the gap is small; source signature does not appear to dominate.")

native resolution in the TRAINING split — what the model learned from:
  class         512x512   other
  glioma           1400       0
  meningioma       1267     133
  notumor            17    1383
  pituitary        1328      72

Read that table before anything else. Every single training glioma is 512x512,
and almost every training notumor is not. So on the data the model was given,
"this file is not 512x512" implies "notumor" with essentially no exceptions.

That reframes the problem. The network did not fail to learn the rule we wanted;
it learned the rule that was actually optimal on the training set. A shortcut is
not a bug in the model, it is a property of the data.

It also explains why validation never caught this. The validation split is
carved out of Training/, so it inherits the same clean correlation and scores
the shortcut as correct every time. Only the test set, assembled differently,
contains tumour scans in the other style.

native resolution in the TEST split:
  cla

  saved -> outputs/shortcut_audit.png

largest per-class gap: 0.765

-> CONFIRMED. The model reads source signature, not anatomy, and the headline
   accuracy overstates what it knows about tumours.

   Two things follow, and the second is the one worth putting in the report.

   First, this is not fixable by augmentation. An augmentation can teach
   invariance to a nuisance variable the model has seen vary; it cannot invent
   training examples of a class in a style that occurs zero times under that
   label. A resample augmentation was tried specifically to destroy the blur
   signature and moved the glioma gap not at all — a negative result, and the
   right one, because blur was never the mechanism. The mechanism is a missing
   region of the training distribution.

   Second, no amount of held-out evaluation on this dataset would have caught
   it. Train, validation and test all inherit the same correlation. It was
   found by looking at the pictures — Grad-CAM pointing at the sk

In [13]:
# 12. SUMMARY AND HONEST LIMITATIONS
"""
The limitations below are not boilerplate. Each one is a specific reason the
number above could be optimistic, and naming them is what separates a result
from a claim.
"""
print("=" * 64)
print("NOTEBOOK 03 — RESULTS")
print("=" * 64)
macro_f1 = [r for r in rows if r["class"] == "macro avg"][0]["f1"]
print(f"  test accuracy        {acc:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
print(f"  macro F1             {macro_f1:.4f}")
print(f"  macro AUC            {macro_auc:.4f}")
print(f"  weakest class        {worst['class']} (recall {worst['recall']:.4f})")
print(f"  errors               {n_wrong} of {len(y_true)}")
print(f"  accuracy on native 512x512      {correct[is512].mean():.4f}")
print(f"  accuracy on every other size    {correct[~is512].mean():.4f}   <- section 11")

checks = [
    ("every test image scored", len(y_true) == 1497),
    ("accuracy above chance", acc > 0.5),
    ("accuracy not implausibly high", acc <= 0.995),
    ("all four classes have non-zero recall",
     all(r["recall"] > 0 for r in rows[:len(CLASSES)])),
    ("figures written", all((config.OUTPUTS / f).exists() for f in
        ["confusion_matrix.png", "roc_curves.png", "confidence.png",
         "feature_maps.png", "grad_cam.png", "shortcut_audit.png"])),
]
print("\n  verification:")
for label, ok in checks:
    print(f"    {'OK  ' if ok else 'FAIL'}  {label}")

print("""
  LIMITATIONS

  1. Source-signature shortcut, measured in section 11. The headline
     accuracy is inflated because the three source collections merged into
     this dataset have different native resolutions that correlate almost
     perfectly with the label. The test split inherits that correlation, so
     the shortcut keeps paying here and the number above overstates what the
     model knows about tumours. This is the most important limitation and,
     unlike the rest, it was demonstrated rather than merely suspected.

  2. No patient identifiers. This dataset ships none, so slices from one
     patient may sit in both Training/ and Testing/. Neighbouring slices of
     the same brain are near duplicates, which would inflate every number
     above, and it cannot be measured with the data as published.

  3. Pre-augmented images. The meningioma class was padded by the dataset
     author with transformed copies. The 103 in Testing/ were removed here;
     the 100 remaining in Training/ are harmless but mean meningioma had
     slightly less genuine variety to learn from than the other classes.

  4. Known label noise. The glioma folder in the source data has been
     reported as partly mis-categorised. Some errors in section 8 may be
     correct predictions against wrong labels.

  5. Single split, single seed. One test set and one training run. The
     confidence intervals in section 5 capture sampling variation in the test
     set, not variation between training runs.

  6. Not a clinical result. Retrospective JPEGs of unknown provenance,
     scored offline, with no comparison against radiologist performance and
     no prospective validation. Given limitation 1, this model should not be
     described as a tumour classifier without qualification.""")

NOTEBOOK 03 — RESULTS
  test accuracy        0.9439   95% CI [0.9325, 0.9309]
  macro F1             0.9418
  macro AUC            0.9866
  weakest class        glioma (recall 0.8300)
  errors               84 of 1497
  accuracy on native 512x512      0.9800
  accuracy on every other size    0.8896   <- section 11

  verification:
    OK    every test image scored
    OK    accuracy above chance
    OK    accuracy not implausibly high
    OK    all four classes have non-zero recall
    OK    figures written

  LIMITATIONS

  1. Source-signature shortcut, measured in section 11. The headline
     accuracy is inflated because the three source collections merged into
     this dataset have different native resolutions that correlate almost
     perfectly with the label. The test split inherits that correlation, so
     the shortcut keeps paying here and the number above overstates what the
     model knows about tumours. This is the most important limitation and,
     unlike the rest, it 